# Lung fibrosis (GSE250346): s carries lesion context and the gradient toward fibrotic lesions

Unaffected vs fibrotic lung cores (Xenium, 343 genes). Ground truth for intra-fibrosis heterogeneity is the
pathologist H&E region annotation released with GSE250346. All z/s comparisons use fibrosis-arm cells only.

Main results: within fibrotic cores s separates H&E regions better than z in 75/109 region pairs; along the distance
to the nearest lesion, s predicts proximity better than z in 94/139 (supervised) cell type × core strata. Strongest
case: alveolar fibroblasts approaching fibroblastic foci (spatial-shift null p = 0.005; collagen/POSTN/CTHRC1 up).
Disease stage (Less vs More Affected) is carried by z, not s.

In [ ]:
import sys, pathlib; sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), pathlib.Path.cwd() / 'notebooks'] if (p / 'repro.py').exists())))
from repro import run, require, show, fetch, fetch_fitted, STAGES, H, WK, PY, PY_VIO, GPU
# Default: download inputs + the paper's fitted Haruka outputs from Hugging Face, then analyses -> figure.
# The Haruka fit step reuses the unpacked fit; HARUKA_REFIT=1 (env) refits from scratch. Auxiliary fits (baselines, seeds) still run.
# Redraw only from saved outputs: STAGES.update(download=False, fit=False, analysis=False)  (or env HARUKA_FIGURE_ONLY=1)
# Rebuild the inputs from the raw public releases instead of Hugging Face: STAGES.update(data=True)
print(H, STAGES)

## 0 Preprocessed data (Hugging Face)

The ready-to-fit inputs are read directly from [C1vy/haruka_v2_data](https://huggingface.co/datasets/C1vy/haruka_v2_data) (cached in `hf_data/`) and linked to the paths the scripts read. The data stage below rebuilds the same files from the raw releases and is off by default.

In [ ]:
fetch('06_fibrosis/gse250346_unaffected_vs_all_fibrosis.h5ad', H / 'bench_lung_fibrosis_gse250346/data/gse250346_unaffected_vs_all_fibrosis.h5ad')
fetch('06_fibrosis/he_annot_matched.csv.gz', WK / 'haruka_fibrosis_he/he_annot_matched.csv.gz')

The paper's default Haruka fit, published as fitted AnnData (input + z, s, predicted neighbourhoods; see the dataset card). It is unpacked into the fit folder, so the fit step below reuses it. Set `HARUKA_REFIT=1` before launching to refit instead.

In [ ]:
fetch_fitted('fitted/06_fibrosis/gse250346_unaffected_vs_all_fibrosis_haruka_fitted.h5ad', H / 'results_real_compgated_nb_final/gse250346_unaffected_vs_all_fibrosis/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0')

## 1 Data: download, export from the Seurat RDS (R), build the h5ad

In [ ]:
run(f'wget -c -O bench_lung_fibrosis_gse250346/data/GSE250346_corrected.rds https://ftp.ncbi.nlm.nih.gov/geo/series/GSE250nnn/GSE250346/suppl/GSE250346_Seurat_GSE250346_CORRECTED_SEE_RDS_README_082024.rds', 'data')
run(f'Rscript bench_lung_fibrosis_gse250346/export_gse250346_unaffected_vs_fibrosis.R bench_lung_fibrosis_gse250346/data/GSE250346_corrected.rds bench_lung_fibrosis_gse250346/data/export_unaffected_vs_fibrosis', 'data')
run(f'{PY} bench_lung_fibrosis_gse250346/build_h5ad.py --export-dir bench_lung_fibrosis_gse250346/data/export_unaffected_vs_fibrosis --out bench_lung_fibrosis_gse250346/data/gse250346_unaffected_vs_all_fibrosis.h5ad --contrast unaffected-vs-fibrosis', 'data')

## 2 H&E annotations (GEO supplementary) matched to cells by barcode + centroid

In [ ]:
HE = WK / 'haruka_fibrosis_he'
run(f'mkdir -p {HE}/raw {HE}/x && cd {HE}/raw && curl -sSLO https://ftp.ncbi.nlm.nih.gov/geo/series/GSE250nnn/GSE250346/suppl/GSE250346_HE_annotations.tar.gz && tar -xzf GSE250346_HE_annotations.tar.gz -C ../x', 'data')
run(f'{PY} bench_lung_fibrosis_gse250346/match_he_annotations.py', 'data')

## 3 Fit (default model; batch = core; ~6.8 h on one GPU)

In [ ]:
run('bash bench_real_discovery/run_all_compgated_samplebatch_sepheads.sh large_lung', 'fit')

In [ ]:
require(H / 'results_real_compgated_nb_final/gse250346_unaffected_vs_all_fibrosis/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/background.npy', H / 'results_real_compgated_nb_final/gse250346_unaffected_vs_all_fibrosis/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/salient.npy', WK / 'haruka_fibrosis_he/he_annot_matched.csv.gz')

## 4 Analyses (`zs_stage/`)
* `he_region_zs.py`: per H&E region pair, separability by z vs s (panel e)
* `he_region_spatial.py`: the same against a spatial-shift null (panel f)
* `lesion_gradient.py`: supervised proximity-to-lesion gradient, z vs s per cell type × core (panel g)
* `lesion_gradient_unsup.py`: unsupervised PC1/PC2/DPT axes vs lesion distance (panel h)
* `lesion_cases.py`: deep dive on the strongest cases, genes and spatial-shift nulls (panels j, k)

In [ ]:
Z = 'bench_lung_fibrosis_gse250346/zs_stage'
run(f'{PY} he_region_zs.py', 'analysis', cwd=H / Z, log='he_region/run.out')
run(f'{PY} he_region_spatial.py', 'analysis', cwd=H / Z, log='he_region/spatial.out')
run(f'{PY} lesion_gradient.py', 'analysis', cwd=H / Z, log='lesion_gradient/run.out')
run(f'{PY} lesion_gradient_unsup.py', 'analysis', cwd=H / Z, log='lesion_gradient/unsup.out')
run(f'mkdir -p lesion_gradient/cases && {PY} lesion_cases.py', 'analysis', cwd=H / Z, log='lesion_gradient/cases/run.out')

## 5 Figure data (example core TILD117MA1, global UMAPs) and the figure

In [ ]:
F = H / 'figures_paper/fibrosis'
run(f'{PY} fib_fig_data.py', 'analysis', cwd=F, env={'EXAMPLE_CORE': 'TILD117MA1'})
run(f'{PY} fib_umap_data.py', 'analysis', cwd=F)
run(f'{PY} make_fibrosis_figure.py', 'figure', cwd=F)

## Figure

In [ ]:
show(H / 'figures_paper/fibrosis/Fibrosis_lesion_gradient_full.png')